In [70]:
import pandas as pd 
import numpy as np
import calendar

In [71]:
df = pd.read_parquet('../Processed_data/Superstore_Processed.parquet')
df

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Year,Month,Delivery Days,Discount Level
0,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.00,41.91,2016,11,3,None
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,...,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.00,219.58,2016,11,3,None
2,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,...,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.00,6.87,2016,6,4,None
3,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,Tables,Bretford CR4500 Series Slim Rectangular Table,957.58,5,0.45,-383.03,2015,10,7,High
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,...,Storage,Eldon Fold 'N Roll Cart System,22.37,2,0.20,2.52,2015,10,7,Standard
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9988,CA-2014-110422,2014-01-21,2014-01-23,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,Florida,...,Furnishings,Ultra Door Pull Handle,25.25,3,0.20,4.10,2014,1,2,Standard
9989,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,...,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.96,2,0.00,15.63,2017,2,5,None
9990,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,...,Phones,Aastra 57i VoIP phone,258.58,2,0.20,19.39,2017,2,5,Standard
9991,CA-2017-121258,2017-02-26,2017-03-03,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,...,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.60,4,0.00,13.32,2017,2,5,None


#### 1. How is the business doing overall?

In [72]:
total_sales = df['Sales'].sum()

print(f"Total sales equals: ${total_sales:,.2f}")

total_profit = df['Profit'].sum()

print(f"Total profit equals: ${total_profit:,.2f}")

print(f"The profit margin equals: {(total_profit / total_sales)*100:,.2f}%")

total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()

print(f"Total number of orders in the dataset equals: {total_orders:,} distributed over {total_customers:,} customers")

total_order_value = df.groupby('Order ID')['Sales'].sum()
average_order_value = total_order_value.mean()

print(f"The average value of orders in our dataset is: ${average_order_value:,.2f}")

Total sales equals: $2,296,919.28
Total profit equals: $286,408.60
The profit margin equals: 12.47%
Total number of orders in the dataset equals: 5,009 distributed over 793 customers
The average value of orders in our dataset is: $458.56


### Q1 Insight

Over the four years in the data (2014 to 2017), the store generated $2,296,919.28 in sales and $286,408.60 in profit, a profit margin of 12.47%. In other words, about 12 to 13 dollars of every 100 dollars sold stays as profit.

The 5,009 orders came from 793 customers, which is about 6 orders per customer, so customers tend to come back. The average order value is $458.56.

These totals are the reference line for the other questions. Later results (by category, region, segment, and discount level) will be compared against the overall margin of 12.47% to see which parts of the business are above or below it.

### Q1 Reference Numbers

| KPI | Value |
|---|---|
| Total Sales | $2,296,919.28 |
| Total Profit | $286,408.60 |
| Profit Margin | 12.47% |
| Total Orders | 5,009 |
| Total Customers | 793 |
| Average Order Value | $458.56 |

These values will be used to check the KPI cards in the Power BI dashboard.

#### 2. Is the business growing?

In [73]:
sales_by_year = df.groupby('Year')['Sales'].sum()
print("Total sales in each year:")
for year, value in sales_by_year.items():
    print(f"- {year}: ${value:,.2f}")

print(30*' =')

profit_by_year = df.groupby('Year')['Profit'].sum()
print("Total profit in each year:")
for year, value in profit_by_year.items():
    print(f"- {year}: ${value:,.2f}")

print(30*' =')

distinct_years = sorted(df['Year'].unique())
year_over_year_profit_growth = {}

for year in distinct_years:
    if year == min(distinct_years):
        year_over_year_profit_growth[year] = "insufficient data"
    else:    
        value = ((profit_by_year[year] - profit_by_year[year-1]) / profit_by_year[year-1]) * 100
        year_over_year_profit_growth[year] = value

print("The Year over Year profit growth for every year is:")
for year , value in year_over_year_profit_growth.items():
    if year == min(distinct_years):
        print(f"- There was insufficient data to get the yoy growth for {year}")
    else:    
        print(f"- {year} --> {value:.2f}%")

print(30*' =')

year_over_year_sales_growth = {}

for year in distinct_years:
    if year == min(distinct_years):
        year_over_year_sales_growth[year] = "insufficient data"
    else:    
        value = ((sales_by_year[year] - sales_by_year[year-1]) / sales_by_year[year-1]) * 100
        year_over_year_sales_growth[year] = value

print("The Year over Year sales growth for every year is:")
for year , value in year_over_year_sales_growth.items():
    if year == min(distinct_years):
        print(f"- There was insufficient data to get the yoy growth for {year}")
    else:    
        print(f"- {year} --> {value:.2f}%")


Total sales in each year:
- 2014: $483,966.10
- 2015: $470,532.39
- 2016: $609,205.74
- 2017: $733,215.05
 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =
Total profit in each year:
- 2014: $49,555.92
- 2015: $61,618.44
- 2016: $81,794.90
- 2017: $93,439.34
 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =
The Year over Year profit growth for every year is:
- There was insufficient data to get the yoy growth for 2014
- 2015 --> 24.34%
- 2016 --> 32.74%
- 2017 --> 14.24%
 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =
The Year over Year sales growth for every year is:
- There was insufficient data to get the yoy growth for 2014
- 2015 --> -2.78%
- 2016 --> 29.47%
- 2017 --> 20.36%


In [74]:
print("Profit margin in each year:")
for year in distinct_years:
    margin = profit_by_year[year] / sales_by_year[year] * 100
    print(f"{year}: {margin:.2f}%")

Profit margin in each year:
2014: 10.24%
2015: 13.10%
2016: 13.43%
2017: 12.74%


In [75]:
#checking the truth of the data we got answering the second business question

print(round(sales_by_year.sum(), 2) == round(total_sales, 2))
print(round(profit_by_year.sum(), 2) == round(total_profit, 2))

True
True


### Q2 Insight

Sales and profit by year, with year-over-year growth and margin:

| Year | Sales | Sales Growth | Profit | Profit Growth | Profit Margin |
|---|---|---|---|---|---|
| 2014 | $483,966.10 | n/a | $49,555.92 | n/a | 10.24% |
| 2015 | $470,532.39 | -2.78% | $61,618.44 | +24.34% | 13.10% |
| 2016 | $609,205.74 | +29.47% | $81,794.90 | +32.74% | 13.43% |
| 2017 | $733,215.05 | +20.36% | $93,439.34 | +14.24% | 12.74% |

The business grew over the four years. Sales rose from about $484K in 2014 to about $733K in 2017, and profit nearly doubled, from about $50K to about $93K. 2014 has no growth rate because there is no earlier year to compare it with.

Three things stand out:

- **2015 is the odd year.** Sales dipped slightly (-2.78%), yet profit grew by about a quarter (+24.34%). The margin jumped from about 10% to 13%, so the store earned more from fewer sales.
- **2016 and 2017 show strong sales growth** (+29.47% and +20.36%), with profit growing at a similar pace in 2016 and more slowly in 2017.
- **In 2017 the margin slipped** from 13.43% to 12.74%. Profit grew slower (+14.24%) than sales (+20.36%), which suggests the business is getting bigger but keeping a little less of each dollar sold.

The data shows what changed, but not why. Possible explanations, such as discounts or the product mix, are tested in later questions (especially Q6 on discounts).

Check: the yearly sales add up to $2,296,919.28 and the yearly profits to $286,408.60, matching the Q1 totals.

#### 3. Are there seasonal patterns?

In [76]:
sales_by_month = df.groupby('Month')['Sales'].sum()

print("Total sales in each month (all years combined):")
for month, value in sales_by_month.items():
    print(f"{calendar.month_name[month]}: ${value:,.2f}")

print(round(sales_by_month.sum(), 2) == round(total_sales, 2))

Total sales in each month (all years combined):
January: $94,924.87
February: $59,751.26
March: $205,005.48
April: $137,480.76
May: $155,028.82
June: $152,718.64
July: $147,238.07
August: $159,043.98
September: $307,649.90
October: $200,322.97
November: $352,461.03
December: $325,293.50
True


In [77]:
print("Top 3 months:")
for month, value in sales_by_month.nlargest(3).items():
    print(f"{calendar.month_name[month]}: ${value:,.2f}")

print(30*' =')

print("Bottom 3 months:")
for month, value in sales_by_month.nsmallest(3).items():
    print(f"{calendar.month_name[month]}: ${value:,.2f}")

Top 3 months:
November: $352,461.03
December: $325,293.50
September: $307,649.90
 = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =
Bottom 3 months:
February: $59,751.26
January: $94,924.87
April: $137,480.76


In [78]:
monthly_by_year = df.pivot_table(index='Month', columns='Year', values='Sales', aggfunc='sum')

print(monthly_by_year.isna().sum().sum())
print(round(monthly_by_year.sum().sum(), 2) == round(total_sales, 2))

share_by_year = monthly_by_year / monthly_by_year.sum() * 100

for year in share_by_year.columns:
    top3 = share_by_year[year].nlargest(3).index
    bottom3 = share_by_year[year].nsmallest(3).index
    print(f"{year} busiest: {[calendar.month_abbr[m] for m in top3]}, slowest: {[calendar.month_abbr[m] for m in bottom3]}")

share_table = share_by_year.copy()
share_table.index = [calendar.month_abbr[m] for m in share_table.index]
share_table.round(1)

0
True
2014 busiest: ['Sep', 'Nov', 'Dec'], slowest: ['Feb', 'Jan', 'May']
2015 busiest: ['Nov', 'Dec', 'Sep'], slowest: ['Feb', 'Jan', 'Jun']
2016 busiest: ['Dec', 'Nov', 'Sep'], slowest: ['Jan', 'Feb', 'Aug']
2017 busiest: ['Nov', 'Sep', 'Dec'], slowest: ['Feb', 'Apr', 'Jan']


Year,2014,2015,2016,2017
Jan,2.9,3.9,3.0,6.0
Feb,0.9,2.5,3.8,2.8
Mar,11.5,8.2,8.5,8.0
Apr,5.8,7.3,6.4,5.0
May,4.9,6.4,9.4,6.0
Jun,7.1,5.3,6.6,7.2
Jul,7.0,6.1,6.4,6.2
Aug,5.8,7.8,5.1,8.6
Sep,16.9,13.7,12.1,12.0
Oct,6.5,6.7,9.8,10.6


### Q3 Insight

Sales follow a clear seasonal pattern that repeats every year.

- **Busiest months:** September, November and December are the three busiest months in every year from 2014 to 2017 (only their order changes). Together they bring between about 40% and 48% of a year's sales, while an even spread would give them about 25%.
- **Slowest months:** January and February are among the three slowest months in every year. The third slowest month changes from year to year, so it is not a stable pattern.
- **The pattern is softening:** the share of sales from September, November and December fell from 47.5% in 2014 to 39.6% in 2017, while January and February rose from 3.8% to 8.8%. Sales are becoming more evenly spread over the year, helped by a stronger October (6.5% in 2014 to 10.6% in 2017).
- **One-off:** March 2014 had an unusually high share (11.5%) that did not repeat in later years.

Why this matters: the business can plan stock, staff and promotions around the autumn and year-end peak, and consider ways to lift sales in January and February.

The data shows when sales peak, but not why. Explanations such as holiday shopping are guesses that this dataset does not test.

Checks: the 12 monthly totals add up to $2,296,919.28, and no month is missing in any year.

#### 4. Which categories and sub-categories bring the most sales and profit?

In [79]:
by_category = df.groupby('Category')[['Sales', 'Profit']].sum()
by_category['Margin %'] = by_category['Profit'] / by_category['Sales'] * 100
by_category = by_category.sort_values('Profit', ascending=False)

print(round(by_category['Sales'].sum(), 2) == round(total_sales, 2))
print(round(by_category['Profit'].sum(), 2) == round(total_profit, 2))

by_category.round(2)

True
True


,Sales,Profit,Margin %
Category,,,
Technology,836154.02,145455.44,17.40
Office Supplies,719046.90,122490.00,17.04
Furniture,741718.36,18463.16,2.49


In [80]:
by_subcategory = df.groupby('Sub-Category')[['Sales', 'Profit']].sum()
by_subcategory['Margin %'] = by_subcategory['Profit'] / by_subcategory['Sales'] * 100
by_subcategory = by_subcategory.sort_values('Profit', ascending=False)

print(round(by_subcategory['Sales'].sum(), 2) == round(total_sales, 2))
print(round(by_subcategory['Profit'].sum(), 2) == round(total_profit, 2))

by_subcategory.round(2)

True
True


,Sales,Profit,Margin %
Sub-Category,,,
Copiers,149528.01,55617.88,37.20
Phones,330007.10,44516.06,13.49
Accessories,167380.31,41936.78,25.05
Paper,78479.24,34053.11,43.39
Binders,203412.68,30221.48,14.86
Chairs,328167.73,26602.14,8.11
Storage,223843.59,21278.90,9.51
Appliances,107532.14,18137.99,16.87
Furnishings,91705.12,13059.18,14.24


### Q4 Insight

Sales, profit and margin by category (the overall margin is 12.47%):

| Category | Sales | Profit | Margin |
|---|---|---|---|
| Technology | $836,154.02 | $145,455.44 | 17.40% |
| Office Supplies | $719,046.90 | $122,490.00 | 17.04% |
| Furniture | $741,718.36 | $18,463.16 | 2.49% |

- **Technology** is first in both sales and profit. It brings 36.4% of the sales and 50.8% of the profit.
- **Office Supplies** has the lowest sales (31.3% of the total) but the second-highest profit (42.8% of the total).
- **Furniture** is the weak spot. It is second in sales (32.3% of the total) but earns only 6.4% of the profit, with a margin of 2.49%, far below the overall 12.47%. Technology and Office Supplies are both above it.

Sub-categories:

- **Most profitable:** Copiers ($55,617.88, margin 37.20%), Phones ($44,516.06, 13.49%) and Accessories ($41,936.78, 25.05%).
- **Highest sales:** Phones ($330,007.10) and Chairs ($328,167.73). Chairs keep only 8.11% of their sales as profit.
- **Machines** sold $189,238.60 but earned only $3,384.72 (margin 1.79%).
- **Furniture's weak result comes from two sub-categories.** Chairs and Furnishings earn $39,661.32 together, while Tables and Bookcases lose $21,198.16 (see Q5).

Sales alone can be misleading: a group with big sales can still keep very little profit, which is why margin is shown next to profit. The data shows where the profit comes from, but not why the margins differ, because product costs are not in the dataset.

Checks: category and sub-category sales add up to $2,296,919.28, and profits add up to $286,408.60.

#### 5. Which sub-categories lose money?

In [81]:
losing_subcategories = by_subcategory[by_subcategory['Profit'] < 0].sort_values('Profit')

print(f"Sub-categories with negative total profit: {len(losing_subcategories)}")

losing_subcategories.round(2)

Sub-categories with negative total profit: 3


,Sales,Profit,Margin %
Sub-Category,,,
Tables,206965.53,-17725.57,-8.56
Bookcases,114879.98,-3472.59,-3.02
Supplies,46673.52,-1189.08,-2.55


In [82]:
is_losing = df['Sub-Category'].isin(losing_subcategories.index)

print(f"Average discount in loss-making sub-categories: {df.loc[is_losing, 'Discount'].mean() * 100:.2f}%")
print(f"Average discount in all other sub-categories: {df.loc[~is_losing, 'Discount'].mean() * 100:.2f}%")

Average discount in loss-making sub-categories: 19.82%
Average discount in all other sub-categories: 15.28%


### Q5 Insight

Three sub-categories lose money overall:

| Sub-category | Sales | Profit | Margin |
|---|---|---|---|
| Tables | $206,965.53 | -$17,725.57 | -8.56% |
| Bookcases | $114,879.98 | -$3,472.59 | -3.02% |
| Supplies | $46,673.52 | -$1,189.08 | -2.55% |

- Together they sold $368,519.03 and lost $22,387.24 (a combined margin of -6.07%).
- Tables alone account for about 79% of that loss.
- Tables and Bookcases both belong to Furniture, which explains the category's low margin in Q4. Supplies belongs to Office Supplies.
- Machines (margin 1.79%, Q4) is not losing money, but it comes close to break-even.

**Discount:** the average discount in the three loss-making sub-categories is 19.82%, compared with 15.28% in all the other sub-categories, about 4.5 percentage points higher. This suggests that heavy discounting may be part of the reason, which Q6 tests directly. It is only a hint, since the figures are simple averages across rows.

This looks at each sub-category's total profit. Individual orders with a loss also exist inside profitable sub-categories, which is normal. The data shows which groups lose money, but not why, because costs are not in the dataset.

#### 6. Do discounts hurt profit?

In [83]:
by_discount = df.groupby('Discount Level', observed=True)[['Sales', 'Profit']].sum()
by_discount['Margin %'] = by_discount['Profit'] / by_discount['Sales'] * 100
by_discount['Rows'] = df['Discount Level'].value_counts()
by_discount = by_discount.reindex(['None', 'Standard', 'Medium', 'High'])

print(round(by_discount['Sales'].sum(), 2) == round(total_sales, 2))
print(by_discount['Rows'].sum() == len(df))

by_discount.round(2)

True
True


,Sales,Profit,Margin %,Rows
Discount Level,,,,
None,1087908.47,320987.01,29.50,4798
Standard,846522.11,100785.81,11.91,3803
Medium,233856.58,-35805.48,-15.31,459
High,128632.12,-99558.74,-77.40,933


In [84]:
by_discount_value = df.groupby('Discount')[['Sales', 'Profit']].sum()
by_discount_value['Margin %'] = by_discount_value['Profit'] / by_discount_value['Sales'] * 100
by_discount_value['Rows'] = df['Discount'].value_counts()

by_discount_value.round(2)

,Sales,Profit,Margin %,Rows
Discount,,,,
0.00,1087908.47,320987.01,29.50,4798
0.10,54369.30,9029.21,16.61,94
0.15,27558.53,1418.98,5.15,52
0.20,764594.28,90337.62,11.82,3657
0.30,102945.30,-10357.26,-10.06,226
0.32,14493.45,-2391.16,-16.50,27
0.40,116417.83,-23057.06,-19.81,206
0.45,5484.98,-2493.12,-45.45,11
0.50,58918.53,-20506.49,-34.80,66


### Q6 Insight

Profit by discount level:

| Discount level | Rows | Sales | Profit | Margin |
|---|---|---|---|---|
| None (0%) | 4,798 | $1,087,908.47 | $320,987.01 | 29.50% |
| Standard (up to 20%) | 3,803 | $846,522.11 | $100,785.81 | 11.91% |
| Medium (20% to 40%) | 459 | $233,856.58 | -$35,805.48 | -15.31% |
| High (above 40%) | 933 | $128,632.12 | -$99,558.74 | -77.40% |

- **Direction:** the higher the discount, the lower the margin, at every step: 29.50%, 11.91%, -15.31%, then -77.40%.
- **Where it turns negative:** by exact discount value, the margin is still positive at 20% (11.82%) and negative from 30% (-10.06%). So the break-even point lies between 20% and 30%. Every discount of 30% or more has a negative margin.
- **Standard** is mostly the usual 20% promotion (3,657 of its 3,803 rows), and its margin of 11.91% is close to the overall 12.47%.
- **Size of the problem:** the Medium and High levels together are 1,392 rows (13.9% of all rows) and $362,488.70 in sales (15.8% of all sales), yet they lost $135,364.22. That loss equals about 47% of the final total profit of $286,408.60.
- **Full-price sales** brought in 47.4% of the sales and earned $320,987.01, which is more than the total profit of the whole business. All discounted orders together lost $34,578.41.
- **At the extremes:** with an 80% discount, the loss ($30,539.11) is larger than the sales ($16,963.68), a margin of -180.03%. At 70% the margin is -98.66%.
- Discount values with very few rows (15% with 52 rows, 45% with 11 rows) are too small to read much into.

Why this matters: the data suggests that reviewing discounts above 20% to 30% could protect profit.

This shows that heavy discounts and low or negative margins go together, but it does not prove that discounts alone cause the losses. Costs are not in the dataset, and the product mix (see Q5) may also play a part.

Checks: the levels add up to $2,296,919.28 in sales and 9,993 rows. The level boundaries were chosen before looking at profit.

#### 7. Which regions and states perform best and worst?

In [85]:
by_region = df.groupby('Region')[['Sales', 'Profit']].sum()
by_region['Margin %'] = by_region['Profit'] / by_region['Sales'] * 100
by_region = by_region.sort_values('Profit', ascending=False)

print(round(by_region['Sales'].sum(), 2) == round(total_sales, 2))
print(round(by_region['Profit'].sum(), 2) == round(total_profit, 2))

by_region.round(2)

True
True


,Sales,Profit,Margin %
Region,,,
West,725457.76,108418.31,14.94
East,678499.93,91534.56,13.49
South,391721.83,46749.49,11.93
Central,501239.76,39706.24,7.92


In [86]:
by_state = df.groupby('State')[['Sales', 'Profit']].sum()
by_state['Margin %'] = by_state['Profit'] / by_state['Sales'] * 100
by_state = by_state.sort_values('Profit', ascending=False)

print(f"Number of states: {len(by_state)}")
print(round(by_state['Sales'].sum(), 2) == round(total_sales, 2))
print(round(by_state['Profit'].sum(), 2) == round(total_profit, 2))

Number of states: 49
True
True


In [87]:
print("Top 10 states by profit:")
display(by_state.head(10).round(2))

print("Bottom 10 states by profit:")
display(by_state.tail(10).round(2))

Top 10 states by profit:


,Sales,Profit,Margin %
State,,,
California,457687.62,76381.23,16.69
New York,310876.20,74038.47,23.82
Washington,138641.29,33402.64,24.09
Michigan,76269.61,24463.10,32.07
Virginia,70636.72,18597.99,26.33
Indiana,53555.36,18382.94,34.33
Georgia,49095.84,16250.00,33.10
Kentucky,36591.75,11199.66,30.61
Minnesota,29863.15,10823.21,36.24


Bottom 10 states by profit:


,Sales,Profit,Margin %
State,,,
Oregon,17431.12,-1190.47,-6.83
Florida,89473.68,-3399.26,-3.80
Arizona,35281.94,-3427.89,-9.72
Tennessee,30661.92,-5341.67,-17.42
Colorado,32108.11,-6527.88,-20.33
North Carolina,55603.07,-7490.84,-13.47
Illinois,80166.11,-12607.94,-15.73
Pennsylvania,116511.98,-15560.06,-13.35
Ohio,77976.82,-16959.32,-21.75


In [88]:
losing_states = by_state[by_state['Profit'] < 0].sort_values('Profit')

print(f"States with negative total profit: {len(losing_states)}")

display(losing_states.round(2))

States with negative total profit: 10


,Sales,Profit,Margin %
State,,,
Texas,170187.91,-25729.32,-15.12
Ohio,77976.82,-16959.32,-21.75
Pennsylvania,116511.98,-15560.06,-13.35
Illinois,80166.11,-12607.94,-15.73
North Carolina,55603.07,-7490.84,-13.47
Colorado,32108.11,-6527.88,-20.33
Tennessee,30661.92,-5341.67,-17.42
Arizona,35281.94,-3427.89,-9.72
Florida,89473.68,-3399.26,-3.80


### Q7 Insight

Sales, profit and margin by region (the overall margin is 12.47%):

| Region | Sales | Profit | Margin |
|---|---|---|---|
| West | $725,457.76 | $108,418.31 | 14.94% |
| East | $678,499.93 | $91,534.56 | 13.49% |
| South | $391,721.83 | $46,749.49 | 11.93% |
| Central | $501,239.76 | $39,706.24 | 7.92% |

- **West** is the best region in sales, profit and margin. It brings 31.6% of the sales and 37.9% of the profit.
- **East** is second in all three, and both West and East are above the overall margin.
- **South** is just below the overall margin (11.93%).
- **Central** is third in sales (21.8% of the total), but it earns the least profit (13.9% of the total) and has the lowest margin, 7.92%. It sells more than South but earns less.

States (49 in the data, including the District of Columbia):

- **Profit is concentrated.** California ($76,381.23) and New York ($74,038.47) together earn $150,419.70, which is 52.5% of the total profit, from 33.5% of the sales. New York keeps more of each sale (margin 23.82%) than California (16.69%). Washington is third ($33,402.64, margin 24.09%).
- **10 states lose money overall:**

| State | Sales | Profit | Margin |
|---|---|---|---|
| Texas | $170,187.91 | -$25,729.32 | -15.12% |
| Ohio | $77,976.82 | -$16,959.32 | -21.75% |
| Pennsylvania | $116,511.98 | -$15,560.06 | -13.35% |
| Illinois | $80,166.11 | -$12,607.94 | -15.73% |
| North Carolina | $55,603.07 | -$7,490.84 | -13.47% |
| Colorado | $32,108.11 | -$6,527.88 | -20.33% |
| Tennessee | $30,661.92 | -$5,341.67 | -17.42% |
| Arizona | $35,281.94 | -$3,427.89 | -9.72% |
| Florida | $89,473.68 | -$3,399.26 | -3.80% |
| Oregon | $17,431.12 | -$1,190.47 | -6.83% |

- Together these 10 states sold $705,402.66 (30.7% of all sales) and lost $98,234.65 (a combined margin of -13.93%). The other 39 states earned $384,643.25.
- **The losses are not in small markets.** Texas, Ohio, Pennsylvania and Illinois account for 72% of the combined loss ($70,856.64), and Texas is the third-largest state in sales.
- The low Central margin is probably linked to Texas and Illinois, two of the biggest losers, which belong to that region.
- **The highest margins** belong to smaller states such as Delaware (36.35%), Minnesota (36.24%) and Indiana (34.33%), with sales between about $27K and $54K. Margins on small sales volumes can swing easily, so they should be read together with sales.

The data shows where the profit and the losses are, but not why. Costs are not in the dataset. Discounts (Q6) and product mix (Q5) may play a part, but this analysis does not test them by state. A natural next question is whether the loss-making states received heavier discounts.

Checks: regions and states each add up to $2,296,919.28 in sales and $286,408.60 in profit.

#### 8. Which customer segments are the most valuable, and how fast do we deliver?

In [89]:
by_segment = df.groupby('Segment').agg(
    Sales=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique'),
    Customers=('Customer ID', 'nunique'),
)
by_segment['Margin %'] = by_segment['Profit'] / by_segment['Sales'] * 100
by_segment['Avg Order Value'] = by_segment['Sales'] / by_segment['Orders']
by_segment['Orders per Customer'] = by_segment['Orders'] / by_segment['Customers']
by_segment['Sales per Customer'] = by_segment['Sales'] / by_segment['Customers']
by_segment = by_segment.sort_values('Profit', ascending=False)

print(round(by_segment['Sales'].sum(), 2) == round(total_sales, 2))
print(by_segment['Orders'].sum() == total_orders)
print(by_segment['Customers'].sum() == total_customers)

by_segment.round(2)

True
True
True


,Sales,Profit,Orders,Customers,Margin %,Avg Order Value,Orders per Customer,Sales per Customer
Segment,,,,,,,,
Consumer,1161401.12,134118.64,2586,409,11.55,449.11,6.32,2839.61
Corporate,706146.32,91979.13,1514,236,13.03,466.41,6.42,2992.15
Home Office,429371.84,60310.83,909,148,14.05,472.36,6.14,2901.16


In [90]:
print((df.groupby('Order ID')['Ship Mode'].nunique() > 1).sum())
print((df.groupby('Order ID')['Delivery Days'].nunique() > 1).sum())

0
0


In [91]:
orders = df.drop_duplicates(subset='Order ID')

delivery = orders.groupby('Ship Mode')['Delivery Days'].agg(['count', 'mean', 'min', 'max'])
delivery = delivery.sort_values('mean')

print(delivery['count'].sum() == total_orders)
print(f"Average delivery time: {orders['Delivery Days'].mean():.2f} days")

delivery.round(2)

True
Average delivery time: 3.96 days


,count,mean,min,max
Ship Mode,,,,
Same Day,264,0.05,0,1
First Class,787,2.19,1,4
Second Class,964,3.23,1,5
Standard Class,2994,5.00,3,7


### Q8 Insight

Customer segments:

| Segment | Sales | Profit | Margin | Orders | Customers |
|---|---|---|---|---|---|
| Consumer | $1,161,401.12 | $134,118.64 | 11.55% | 2,586 | 409 |
| Corporate | $706,146.32 | $91,979.13 | 13.03% | 1,514 | 236 |
| Home Office | $429,371.84 | $60,310.83 | 14.05% | 909 | 148 |

| Segment | Average order value | Orders per customer | Sales per customer | Profit per customer |
|---|---|---|---|---|
| Consumer | $449.11 | 6.32 | $2,839.61 | $327.92 |
| Corporate | $466.41 | 6.42 | $2,992.15 | $389.74 |
| Home Office | $472.36 | 6.14 | $2,901.16 | $407.51 |

(Profit per customer is profit divided by the number of customers, calculated from the table above.)

- **Consumer is the biggest segment** in everything: 50.6% of the sales, 46.8% of the profit, 51.6% of the customers (409 of 793) and 51.6% of the orders. But it has the lowest margin, 11.55%, which is below the overall 12.47%.
- **Corporate** brings 30.7% of the sales and 32.1% of the profit, with a margin of 13.03%. **Home Office** is the smallest (18.7% of sales, 21.1% of profit) but has the highest margin, 14.05%. Both are above the overall margin.
- **Customers behave very similarly across segments.** Orders per customer are between 6.14 and 6.42, the average order value differs by about 5% ($449.11 to $472.36), and so does sales per customer. So the size of each segment comes mainly from how many customers it has, not from how each customer buys.
- **Which segment is the most valuable depends on the measure.** In total, it is Consumer. Per customer, it is Home Office ($407.51 profit per customer), followed by Corporate ($389.74) and then Consumer ($327.92).

Delivery time by ship mode (one value per order; the overall average is 3.96 days):

| Ship mode | Orders | Share of orders | Average days | Min | Max |
|---|---|---|---|---|---|
| Same Day | 264 | 5.3% | 0.05 | 0 | 1 |
| First Class | 787 | 15.7% | 2.19 | 1 | 4 |
| Second Class | 964 | 19.2% | 3.23 | 1 | 5 |
| Standard Class | 2,994 | 59.8% | 5.00 | 3 | 7 |

- **Standard Class** is used for about 60% of the orders and takes 5 days on average, up to 7.
- Faster modes deliver faster, as expected, but the ranges overlap: a Second Class order can take up to 5 days, the same as the Standard Class average.
- **Same Day** averages 0.05 days. Almost all orders arrive the same day, and a few took one day.
- These averages are per order, since an order is one delivery. They differ very slightly from the per-row averages in the Prepare stage (for example Standard Class 5.00 against 5.01).

The data does not include shipping cost or customer satisfaction, so it cannot say whether the faster modes are worth their price.

Checks: segments add up to $2,296,919.28 in sales, 5,009 orders and 793 customers. Every order has exactly one ship mode and one delivery time, and the ship-mode counts add up to 5,009.